# SignalShap — End-to-End Reproduction

Runs the full study step by step: **load data → train scorers → build candidates → run the 32-coalition game → E0–E8 → figures & tables**.

Implements `SignalShap_Implementation_Spec.md`. Two rules govern the ordering below and are enforced, not advisory:

| Gate | What it protects | Spec |
|---|---|---|
| **E0-a candidate recall** | The ceiling on *every* ranking metric. A test item outside $C_u$ scores zero by construction. | §2.2 |
| **E0-b monotonicity audit** | Whether Property 2 / Lemma 1(ii) apply at all. | §3.3 |

> **On "exact".** The 32-coalition aggregation has **no sampling error** — unlike Monte-Carlo Shapley. But $v(S)$ is *fitted* (ridge heads on a validation fold), so $\varphi_g$ carries estimation error and is reported with seed CIs **in the main text**. Never "no error"; always "no sampling error." (§2.7)

## 1 · Setup

Installs dependencies and puts `src/` on the path.

In [ ]:
%pip install -q numpy pandas scipy scikit-learn matplotlib pyyaml pyarrow tabulate jinja2 2>/dev/null

import sys, os, json, time, warnings
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent          # running from notebooks/
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd
import matplotlib.pyplot as plt

print("root:", ROOT)

### Configuration

`configs/frozen.yaml` holds everything **pre-registered in Week 1**: per-dataset $N_{max}$, the frozen ridge $\lambda$, and the $v_0$ permutation seed.

**`SYNTHETIC = True`** generates deterministic corpora with *planted* structure (popularity skew, latent factors, content clusters, temporal drift) so the pipeline is runnable before any download. Set `False` once raw files are in `data/raw/`. Every artefact records which mode produced it.

**`FAST = True`** shrinks $N_{max}$ for a quick pass. Turn it off for the real run.

In [ ]:
from signalshap.config import FrozenConfig, SOURCES, SEEDS, write_artefact, read_artefact

SYNTHETIC = True
FAST      = True
DATASETS  = ("ml_1m", "lastfm_2k", "amazon_book")
SEEDS_RUN = (42, 43, 44)

cfg = FrozenConfig.load()
if FAST:
    cfg = FrozenConfig(n_max={"ml_1m": 80, "lastfm_2k": 100, "amazon_book": 120})
cfg.save()

print("sources :", SOURCES)
print("N_max   :", cfg.n_max, "   <- pre-registered per dataset (§2.2)")
print("lambda  :", cfg.ridge_lambda, "  <- FROZEN, never tuned per coalition (§2.4)")
print("v0 seed :", cfg.v0_seed, " <- one frozen permutation, reused everywhere (§2.5)")
print("gate    : recall >=", cfg.recall_gate)

## 2 · Load data and split

Leave-last-out temporal split: last interaction → test, second-to-last → validation, rest → train. Ties broken by `(timestamp, original_record_index)`.

**Density is computed as-used, never quoted from a source paper** (§6.1) — the subsample alone moves Amazon-Book off its published figure before any filtering.

In [ ]:
from signalshap.data.loaders import load_dataset, build_dataset_stats

datasets = {n: load_dataset(n, synthetic=SYNTHETIC, seed=SEEDS_RUN[0]) for n in DATASETS}

rows = []
for n, ds in datasets.items():
    s = ds.stats()
    rows.append({"dataset": n, "users": s["users"], "items": s["items"],
                 "interactions": s["interactions"],
                 "density %": round(s["density"] * 100, 4),
                 "train": s["train"], "valid": s["valid"], "test": s["test"],
                 "synthetic": s["synthetic"]})
display(pd.DataFrame(rows))

### The density-ordering invariant

C3 claims contrast across *dense movies / sparse books / medium music*. That is only true while

$$\rho_{\text{amazon\_book}} < \rho_{\text{lastfm\_2k}} < \rho_{\text{ml\_1m}}$$

holds with a real margin. It matters because the $k$-core recall remedy raises density — the same axis — and at $k{=}10$ it *inverts* the ordering rather than merely blunting it. The check is a **CI test**, not a comment, and it runs against measured statistics carrying `source: "measured"`.

In [ ]:
stats = build_dataset_stats(datasets)
write_artefact("dataset_stats.json", stats)

from signalshap.config import DENSITY_ORDER, DENSITY_MARGIN
d = {n: stats[n]["density"] for n in DENSITY_ORDER}
print("measured densities (sparsest first expected):")
for n in DENSITY_ORDER:
    print(f"  {n:12s} {d[n]*100:.4f}%")

ok = True
for a, b in zip(DENSITY_ORDER, DENSITY_ORDER[1:]):
    r = d[b] / d[a] if d[a] > 0 else 0
    good = d[a] < d[b] and r >= DENSITY_MARGIN
    ok &= good
    print(f"  {b} / {a} = {r:.2f}x  {'OK' if good else 'VIOLATION'} (need >= {DENSITY_MARGIN}x)")
print("\nINVARIANT:", "HOLDS — C3 wording is safe" if ok else "VIOLATED — rewrite C3 or revert the filter")
print("provenance:", stats["_meta"]["source"], "|", stats["_meta"]["corpus_hash"])

## 3 · Train the five base scorers

Trained **once** each, then cached; all 32 coalitions reuse the same matrices. That is what keeps the Shapley sweep on a laptop CPU.

Two overlaps are **pre-registered, not discovered** (§4): `pop`–`cf` (ALS on implicit feedback chases popularity) and `rec`–`ct` (recency is defined over content clusters). RQ2 tests whether Shapley *recovers* known structure — declaring this up front is what stops the headline result being circular.

In [ ]:
from signalshap.scorers.base import train_all_scorers, mask_seen

DEMO = DATASETS[0]
t0 = time.time()
demo_scores = mask_seen(train_all_scorers(datasets[DEMO], seed=42), datasets[DEMO])
print(f"trained 5 scorers on {DEMO} in {time.time()-t0:.1f}s")
for g, m in demo_scores.items():
    finite = np.isfinite(m)
    print(f"  {g:4s} shape={m.shape}  mean={m[finite].mean():+.4f}  std={m[finite].std():.4f}")

## 4 · Build the candidate set

$C_u=\bigcup_g \text{top-}N_g^{(g)}(u)$ — the union of each source's own list, so **every coalition scores the same items**.

The rejected alternative (draw candidates once from the grand-coalition scorer) hands the grand coalition a pool selected in its own favour, inflating $v(\mathcal{G})$ against every $v(S)$. Since Shapley values are built entirely from differences $v(S\cup\{g\})-v(S)$, that bias propagates into every attribution. It survives only as ablation E8-a.

In [ ]:
from signalshap.candidates.builder import build_candidates, candidate_recall

t0 = time.time()
demo_cands = build_candidates(demo_scores, datasets[DEMO].n_users, cfg.n_max[DEMO])
sizes = np.array([len(c) for c in demo_cands])
test_map = dict(zip(datasets[DEMO].test["user"], datasets[DEMO].test["item"]))
rec = candidate_recall(demo_cands, test_map)

print(f"built in {time.time()-t0:.1f}s")
print(f"|C_u|: mean={sizes.mean():.1f} min={sizes.min()} max={sizes.max()} cap={cfg.n_max[DEMO]}")
print(f"candidate recall = {rec:.3f}   gate {'PASS' if rec >= cfg.recall_gate else 'FAIL'}")

# Coalition-independence: C_u must not depend on which coalition is evaluated.
from signalshap.candidates.builder import build_candidates_for_user
subset = {g: demo_scores[g] for g in ("cf", "ct")}
same = all(np.array_equal(build_candidates_for_user(demo_scores, u, cfg.n_max[DEMO]),
                          demo_cands[u]) for u in range(20))
print("coalition-independent for 20 sampled users:", same)

## 5 · The game: 32 coalitions and exact Shapley

$v(S)$ = mean NDCG@10 over $C_u$ minus $v_0$, where $v_0$ is **one frozen permutation** reused everywhere — so $v(\varnothing)=0$ *exactly and per user*, which Property 3 requires.

$\varphi_g$ is invariant to $v_0$ entirely (the offset cancels in every marginal); it exists only so Property 1 reads $\sum_g\varphi_g=v(\mathcal{G})$.

In [ ]:
from signalshap.game.core import (SignalShapGame, exact_shapley, check_efficiency,
                                  monotonicity_audit, all_coalitions)

t0 = time.time()
game = SignalShapGame(demo_scores, demo_cands,
                      dict(zip(datasets[DEMO].valid["user"], datasets[DEMO].valid["item"])),
                      test_map, ridge_lambda=cfg.ridge_lambda,
                      k_ndcg=cfg.k_ndcg, v0_seed=cfg.v0_seed)
v = game.v_all()
print(f"evaluated {len(v)} coalitions in {time.time()-t0:.1f}s on CPU")
print(f"v(empty) = {v[frozenset()]:.2e}   (exactly 0 by construction)")
print(f"v(G)     = {v[frozenset(SOURCES)]:.5f}")
print(f"v_0      = {game.v0:.5f}  (cosmetic offset; phi is invariant to it)")

print("\nv(S) by coalition size:")
for k in range(len(SOURCES) + 1):
    vals = [v[S] for S in all_coalitions() if len(S) == k]
    print(f"  |S|={k}: n={len(vals):2d}  mean={np.mean(vals):+.5f}  max={np.max(vals):+.5f}")

In [ ]:
phi = exact_shapley(v)
eff = check_efficiency(phi, v)

print("Exact Shapley values (32 coalitions, closed form):")
for g in SOURCES:
    print(f"  phi_{g:4s} = {phi[g]:+.6f}")
print(f"\nProperty 1 (efficiency): sum={eff['sum_phi']:.8f}  v(G)={eff['v_grand']:.8f}")
print(f"  |error| = {eff['abs_error']:.2e}   {'PASS' if eff['passes'] else 'FAIL'}")
print("  Holds exactly regardless of fit noise — efficiency is structural.")

### E0-b — the monotonicity audit

Property 2 needs **monotonicity**, and an earlier draft omitted it. Without that hypothesis the property is *false*: a three-player game satisfying exact redundancy with $v(\{g_1\})=1>0$ yields $\varphi_{g_1}=\varphi_{g_2}=-0.4167$.

So monotonicity is **discharged empirically, never asserted**. The coalition-conditional ridge refit of §2.4 is precisely the mechanism that can break it — a head fitted on $S\cup\{g\}$ may score worse than one fitted on $S$. Violations are a **finding**, not a defect to hide.

In [ ]:
audit = monotonicity_audit(v)
print(f"pairs checked      : {audit['pairs_checked']}  (= |G| * 2^(|G|-1))")
print(f"violations         : {audit['violations']}")
print(f"max magnitude      : {audit['max_magnitude']:.6f}")
print(f"sources involved   : {audit['sources_involved']}")
print(f"Property 2 applies : {audit['property2_applicable']}")
if not audit["property2_applicable"]:
    print("\n-> Non-monotone game. Report negative phi_g as SUBSTANTIVE, and restate")
    print("   Lemma 1(ii) under bounded synergy (>= -delta). Decide this NOW (Week 3),")
    print("   not in Week 7 when it collides with the writing pass.")
    for x in audit["detail"][:5]:
        print(f"     v({set(x['S']) or '{}'} + {x['g']}) - v(...) = {x['delta']:+.6f}")

## 6 · Run E0–E8 on all three datasets

Each experiment writes a JSON to `artefacts/` — no number is ever typed into the paper by hand (§14).

In [ ]:
from signalshap.pipeline import run_full_study

t0 = time.time()
study = run_full_study(datasets=DATASETS, synthetic=SYNTHETIC, seeds=SEEDS_RUN)
results, stats = study["results"], study["dataset_stats"]
print(f"\ncomplete in {time.time()-t0:.0f}s")

summary = []
for n, r in results.items():
    a, b = r["e0a_candidates"], r["e0b_monotonicity"]
    fc = r["e4_signalshap_fuse"]["full_catalog"]
    summary.append({
        "dataset": n,
        "recall": round(a["candidate_recall"], 3),
        "gate": "PASS" if a["gate_passes"] else "FAIL",
        "monot. viol": f"{b['violations']}/{b['pairs_checked']}",
        "Prop2": "yes" if b["property2_applicable"] else "no",
        "eff err": f"{r['e1_source_share']['efficiency']['abs_error']:.1e}",
        "fuse NDCG": round(fc["signalshap_fuse"]["ndcg_at_10"], 4),
        "global NDCG": round(fc["global"]["ndcg_at_10"], 4),
    })
display(pd.DataFrame(summary))

### Gate check

Where recall falls below 0.60, §2.2's **four-rung ladder** applies in order: raise $N_{max}$ → report the ceiling and keep only the *relative* LOO-vs-Shapley claim → $k$-core under the density invariant → substitute Gowalla.

Two remedies are **forbidden**: re-tuning the growth schedule (it redistributes budget between sources and cannot create recall the scorers' top-lists lack), and restricting evaluation to users with $\text{test}_u\in C_u$ (it biases the population *differentially by density* and confounds C3).

In [ ]:
for n, r in results.items():
    a = r["e0a_candidates"]
    if not a["gate_passes"]:
        print(f"[{n}] recall {a['candidate_recall']:.3f} < {a['recall_gate']}")
        print(f"      {r.get('gate_warning', '')[:200]}...\n")
    else:
        print(f"[{n}] recall {a['candidate_recall']:.3f} — PASS\n")

## 7 · E1/E2 — LOO vs Shapley

The empirical core of the paper. Property 2 predicts LOO collapses for redundant sources while Shapley does not; Lemma 1 makes that quantitative for $\varepsilon$-redundancy, with the floor $\varphi_{g_1}\ge v(\{g_1\})/|\mathcal{G}|$ — the constant is $1/|\mathcal{G}|$, **never** $1/2$.

In [ ]:
rows = []
for n, r in results.items():
    e2 = r["e2_loo_vs_shapley"]
    ci = r.get("multi_seed", {}).get("ci", {})
    for g in SOURCES:
        rows.append({"dataset": n, "source": g,
                     "LOO": round(e2["loo"][g], 5),
                     "Shapley": round(e2["shapley"][g], 5),
                     "gap": round(e2["gap"][g], 5),
                     "seed std": round(ci.get(g, {}).get("std", 0), 5),
                     "perm p": round(e2["permutation_tests"][g]["p_value"], 4)})
display(pd.DataFrame(rows))

print("Redundancy (Kendall tau) — pop|cf and rec|ct are PRE-REGISTERED:")
for n, r in results.items():
    tau = r["e2_loo_vs_shapley"]["kendall_tau"]
    top = sorted(tau.items(), key=lambda kv: -abs(kv[1]))[:3]
    pre = {k: round(tau.get(k, 0), 3) for k in ("pop|cf", "rec|ct") if k in tau}
    print(f"  {n:12s} strongest={[(k, round(x,3)) for k,x in top]}  pre-registered={pre}")
    print(f"               corr(tau, gap) = {r['e2_loo_vs_shapley']['redundancy_gap_correlation']:+.3f}")

## 8 · E4 — SignalShap-Fuse, full-catalog

**T5 reports full-catalog for every method.** SignalShap-Fuse scores items outside $C_u$ as $-\infty$, so its recall ceiling is a *visible cost* rather than a hidden denominator advantage. Restricting the baseline to $C_u$ was rejected — it handicaps a strong baseline by confining it to a pool built from five other scorers' lists.

$v(S)$ itself stays fixed-candidate: that is required for coalition-independence and is the game's definition. Only *reporting* is full-catalog.

In [ ]:
for n, r in results.items():
    e4 = r["e4_signalshap_fuse"]
    print(f"=== {n} ===")
    display(pd.DataFrame([
        {"method": m, **{k: round(v[k], 5) for k in ("ndcg_at_10", "recall_at_20", "mrr_at_10")}}
        for m, v in e4["full_catalog"].items()]))
    h = e4["holm_bonferroni"]
    print(f"Holm-Bonferroni family m={h['family_size']}, composition={h['family_composition']}")
    for b, t in e4["wilcoxon"].items():
        print(f"  vs {b:22s} dNDCG={t['mean_diff']:+.5f}  raw p={t['p_value']:.4f}  "
              f"Holm p={h['corrected'][b]:.4f}  d_z={t['d_z']:+.3f}")
    print()

## 9 · E5–E8 — robustness, ablation, actionability

In [ ]:
for n, r in results.items():
    rb = r["e5_robustness"]
    print(f"=== {n} ===")
    print("  |C_u| sweep (recall reported PER CELL — the cells are not otherwise comparable):")
    for k, cell in sorted(rb["candidate_size"].items(), key=lambda kv: kv[1]["n_max"]):
        top = max(cell["shapley"], key=cell["shapley"].get)
        print(f"    N_max={cell['n_max']:4d}  recall={cell['candidate_recall']:.3f}  "
              f"v(G)={cell['v_grand']:.5f}  top={top}")
    print("  lambda sensitivity (reported, NEVER used to select):")
    for k, phi_l in sorted(rb["lambda_sensitivity"].items()):
        print(f"    {k:12s} top={max(phi_l, key=phi_l.get)}  "
              f"ranking={sorted(phi_l, key=phi_l.get, reverse=True)}")
    e7 = r["e7_actionability"]
    print(f"  E7: drop '{e7['lowest_shapley_source']}' -> dNDCG={e7['delta']:+.5f} "
          f"(p={e7['wilcoxon']['p_value']:.4f})")
    e8 = r["e8_appendix_b"]
    print(f"  E8-a: v(G) inflation under rejected grand-coalition candidates = "
          f"{e8['v_grand_inflation']:+.5f}\n")

## 10 · Generate paper assets

Figures **F1–F7** and tables **T1–T8**, built *only* from `artefacts/` JSON. Tables emit Markdown (review), LaTeX (Springer template), and CSV.

In [ ]:
from signalshap.plots.assets import generate_all_assets, FIG, TAB

assets = generate_all_assets(results, stats)
print("figures:")
for k, p in assets["figures"].items():
    print(f"  {k}: {p}")
print("\ntables:", sorted({p.stem.split('.')[0] for p in TAB.glob('T*.md')}))

In [ ]:
from IPython.display import Image, display as disp
for f in ["F1_workflow", "F2_shapley_shares", "F3_loo_vs_shapley", "F4_redundancy_heatmap"]:
    print(f)
    disp(Image(filename=str(FIG / f"{f}.png")))

In [ ]:
for f in ["F5_segment_radar", "F6_fuse_gain", "F7_robustness"]:
    print(f)
    disp(Image(filename=str(FIG / f"{f}.png")))

### T2 — the precondition table

Candidate recall bounds every metric in T5–T7; monotonicity violations decide whether Property 2 applies. Inspect both *before* reading any attribution number.

In [ ]:
print((TAB / "T2_datasets.md").read_text())
print((TAB / "T6_loo_vs_shapley.md").read_text()[:1800])

## 11 · Verify the invariants

The tests that keep the specification honest:

- `test_property2_counterexample.py` — pins the monotonicity hypothesis and rejects two plausible-but-wrong formulas: the $\tfrac12 v(\{g_1\})$ floor (overstates by $n/2$) and the $\tfrac12[v(\mathcal{G})-v(\mathcal{G}\setminus\{g_1,g_2\})]$ closed form (gives $-2.25$ vs a true $-0.4167$).
- `test_density_ordering.py` — the C3 contrast, plus a provenance gate so estimates can never be mistaken for measurements.

In [ ]:
import subprocess
print(subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q", "--no-header"],
                     capture_output=True, text=True).stdout[-1500:])

## 12 · Summary

Written to `artefacts/`:

| File | Contents |
|---|---|
| `results_<dataset>.json` | Every E0–E8 number, per dataset |
| `dataset_stats.json` | As-used statistics with measured provenance |
| `study_manifest.json` | Seeds, config, timestamp, synthetic flag |
| `figures/F1–F7.png` | Paper figures at 300 dpi |
| `tables/T1–T8.{md,tex,csv}` | Paper tables |

**Before trusting any of it:** confirm the E0-a recall gate passed, and check the E0-b monotonicity column in T2. Everything downstream is conditional on both.

In [ ]:
print("ARTEFACTS")
for p in sorted(Path("artefacts").rglob("*")):
    if p.is_file():
        print(f"  {p}  ({p.stat().st_size/1024:.1f} KB)")

print("\nHEADLINE (read T2 first)")
for n, r in results.items():
    a, b = r["e0a_candidates"], r["e0b_monotonicity"]
    phi = r["e1_source_share"]["shapley"]
    print(f"  {n}: recall={a['candidate_recall']:.3f} ({'PASS' if a['gate_passes'] else 'FAIL'}), "
          f"monot. viol={b['violations']}/{b['pairs_checked']}, "
          f"top source={max(phi, key=phi.get)}")
print("\nReminder: 'exact' = exact GIVEN THE FITTED v. No sampling error; not no error.")